# Single IV

In [1]:
# high resolution single iv data
import numpy as np

# get data
from superconductivity.evaluation import FileSpec

filespec = FileSpec(
    h5path="OI-25c-09 2025-05-02 unbroken stripline irradiation studies 0.hdf5",
    location="/Users/oliver/Documents/measurement data/25 04 OI-25c-09",
    measurement="vna_frequencies_0.100V",
)

# get keys
from superconductivity.evaluation import KeysSpec
from superconductivity.evaluation import get_keys

keysspec = KeysSpec(
    strip0="vna_",
    strip1="GHz_0.100V",
    remove_key="no_irradiation",
    add_key=[
        ("no_irradiation", 0.0),
    ],
    limits=(None, 0.01),
    norm=1,
    label="nu_GHz",
)

# get traces
from superconductivity.evaluation import TraceSpec
from superconductivity.evaluation import get_traces

tracespec = TraceSpec(
    AmpV=1000,
    AmpI=1000,
    trigger_values=1,
    skip_edges=5,
)

traces = get_traces(
    filespec=filespec,
    keysspec=keysspec,
    tracespec=tracespec,
)

# offset analysis
from superconductivity.evaluation import OffsetSpec
from superconductivity.evaluation import offset_analysis

offsetspec = OffsetSpec(
    Vbins_mV=np.linspace(-0.5, 0.5, 51),
    Ibins_nA=np.linspace(-5.0, 5.0, 51),
    Voffscan_mV=np.linspace(-0.045, 0.045, 451),
    Ioffscan_nA=np.linspace(-0.35, 0.35, 351),
    cutoff_Hz=43.7,
    sampling_Hz=137.0,
)

offsetanalysis = offset_analysis(
    traces=traces,
    spec=offsetspec,
)

# sampling
from superconductivity.evaluation import SamplingSpec
from superconductivity.evaluation import sample

samplingspec = SamplingSpec(
    Vbins_mV=np.linspace(-1.0, 1.0, 32001),
    Ibins_nA=np.linspace(-30.0, 30.0, 2001),
    Voff_mV=offsetanalysis.Voff_mV,
    Ioff_nA=offsetanalysis.Ioff_nA,
    cutoff_Hz=13.7,
    sampling_Hz=13700.0,
    median_bins=3,
    sigma_bins=40.0,
)

exp_v, exp_i = sample(
    traces=traces,
    samplingspec=samplingspec,
)

# save data
np.savez_compressed(
    "single_iv/eva.npz",
    V_mV=exp_v.V_mV,
    I_nA=exp_v.I_nA[0, :],
    Tbath_K=traces.Tsample_K[0],
)

offset_analysis:   0%|          | 0/2 [00:00<?, ?trace/s]

sampling:   0%|          | 0/2 [00:00<?, ?trace/s]

# Temperature Study

In [1]:
# # generate cache
# from superconductivity.utilities import make_cache, save_cache

# cache = make_cache(name="cache", path="temperature/")
# save_cache(cache)

In [2]:
# load cache
from superconductivity.utilities import load_cache, save_cache

cache = load_cache("cache", path="temperature/")

In [3]:
# get data
from superconductivity.evaluation import FileSpec

cache.filespec = FileSpec(
    h5path="OI-25c-09 2025-04-15 unbroken 1.hdf5",
    location="/Users/oliver/Documents/measurement data/25 04 OI-25c-09",
    measurement="temperatures",
)
cache.mkeys = cache.filespec.mkeys()
cache.skeys = cache.filespec.skeys()
_ = save_cache(cache)

In [4]:
# get keys
from superconductivity.evaluation import KeysSpec
from superconductivity.evaluation import get_keys

cache.keysspec = KeysSpec(
    strip0="heater_",
    strip1="muW",
    remove_key="no_heater",
    add_key=[
        ("no_heater", 0.0),
    ],
    limits=(None, None),
    norm=1e1,
    label="Pheater_uW",
)

cache.skeys = get_keys(
    filespec=cache.filespec,
    keysspec=cache.keysspec,
)
_ = save_cache(cache)

In [5]:
# get traces
from superconductivity.evaluation import TraceSpec
from superconductivity.evaluation import get_traces

cache.tracespec = TraceSpec(
    AmpV=1000,
    AmpI=1000,
    trigger_values=1,
    skip_edges=5,
)

cache.traces = get_traces(
    filespec=cache.filespec,
    keysspec=cache.keysspec,
    tracespec=cache.tracespec,
)
_ = save_cache(cache)

In [6]:
# # offset analysis
# from superconductivity.evaluation import OffsetSpec
# from superconductivity.evaluation import offset_analysis
# import numpy as np

# cache.offsetspec = OffsetSpec(
#     Vbins_mV=np.linspace(-0.5, 0.5, 51),
#     Ibins_nA=np.linspace(-5.0, 5.0, 51),
#     Voffscan_mV=np.linspace(-0.045, 0.045, 451),
#     Ioffscan_nA=np.linspace(-0.35, 0.35, 351),
#     cutoff_Hz=13.7,
#     sampling_Hz=137.0,
# )

# cache.offsetanalysis = offset_analysis(
#     traces=cache.traces,
#     spec=cache.offsetspec,
# )
# _ = save_cache(cache)

In [57]:
# sampling
from superconductivity.evaluation import SamplingSpec
from superconductivity.evaluation import sample
import numpy as np

cache.samplingspec = SamplingSpec(
    Vbins_mV=np.linspace(-0.8, 0.8, 3201),
    Ibins_nA=np.linspace(-12.5, 12.5, 3201),
    Voff_mV=cache.offsetanalysis.Voff_mV,
    Ioff_nA=cache.offsetanalysis.Ioff_nA,
    cutoff_Hz=13.7,
    sampling_Hz=1373.0,
    apply_smoothing=True,
    median_bins=5,
    sigma_bins=15.0,
)

cache.samplingspec.keys()

cache.exp_v, cache.exp_i = sample(
    traces=cache.traces,
    samplingspec=cache.samplingspec,
)
_ = save_cache(cache)

sampling:   0%|          | 0/151 [00:00<?, ?trace/s]

In [60]:
# calibrate

from superconductivity.evaluation import CalibrationSpec
from superconductivity.evaluation import calibrate

cache.calibrationspec = CalibrationSpec(
    label="Tbath_K",
    lookup=cache.traces.Tsample_K,
)
cache.cal_v, cache.cal_i = calibrate(
    cache.exp_v,
    cache.exp_i,
    calibrationspec=cache.calibrationspec,
)
_ = save_cache(cache)

In [61]:
# save data
import numpy as np

np.savez_compressed(
    "temperature/eva.npz",
    Vbias_mV=cache.cal_v["V_mV"],
    Ibias_nA=cache.cal_i["I_nA"],
    Tbath_K=cache.cal_v["Tbath_K"],
    Pheater_uW=cache.traces.Pheater_uW,
    dGexp_G0=cache.cal_v["dG_G0"],
    dRexp_R0=cache.cal_i["dR_R0"],
    Iexp_nA=cache.cal_v["I_nA"],
)

# Amplitude Study 18.3 GHz

In [86]:
# # generate cache
# from superconductivity.utilities import make_cache, save_cache

# cache = make_cache(name="cache", path="amp_18.3GHz/")
# save_cache(cache)

In [87]:
# load cache
from superconductivity.utilities import load_cache, save_cache

cache = load_cache("cache", path="amp_18.3GHz/")

In [88]:
# get data
from superconductivity.evaluation import FileSpec

cache.filespec = FileSpec(
    h5path="OI-25c-09 2025-05-02 unbroken stripline irradiation studies 0.hdf5",
    location="/Users/oliver/Documents/measurement data/25 04 OI-25c-09",
    measurement="vna_amplitudes_18.3000GHz",
)
mkeys = cache.filespec.mkeys()
skeys = cache.filespec.skeys()
_ = save_cache(cache)

In [89]:
# get keys
from superconductivity.evaluation import KeysSpec
from superconductivity.evaluation import get_keys

cache.keysspec = KeysSpec(
    strip0="GHz_",
    strip1="V",
    remove_key="no_irradiation",
    add_key=[
        ("no_irradiation", 0.0),
        ("no_irradiation", 0.005),
    ],
    limits=(None, None),
    norm=1e-3,
    label="Aout_mV",
)

cache.skeys = get_keys(
    filespec=cache.filespec,
    keysspec=cache.keysspec,
)
_ = save_cache(cache)

In [90]:
# get traces
from superconductivity.evaluation import TraceSpec
from superconductivity.evaluation import get_traces

cache.tracespec = TraceSpec(
    AmpV=1000,
    AmpI=1000,
    trigger_values=1,
    skip_edges=5,
)

cache.traces = get_traces(
    filespec=cache.filespec,
    keysspec=cache.keysspec,
    tracespec=cache.tracespec,
)
_ = save_cache(cache)

In [91]:
# # offset analysis
# from superconductivity.evaluation import OffsetSpec
# from superconductivity.evaluation import offset_analysis
# import numpy as np

# cache.offsetspec = OffsetSpec(
#     Vbins_mV=np.linspace(-0.5, 0.5, 51),
#     Ibins_nA=np.linspace(-5.0, 5.0, 51),
#     Voffscan_mV=np.linspace(-0.045, 0.045, 451),
#     Ioffscan_nA=np.linspace(-0.35, 0.35, 351),
#     cutoff_Hz=13.7,
#     sampling_Hz=137.0,
# )

# cache.offsetanalysis = offset_analysis(
#     traces=cache.traces,
#     spec=cache.offsetspec,
# )
# _ = save_cache(cache)

In [92]:
# sampling
from superconductivity.evaluation import SamplingSpec
from superconductivity.evaluation import sample
import numpy as np

cache.samplingspec = SamplingSpec(
    Vbins_mV=np.linspace(-1.6, 1.6, 1601),
    Ibins_nA=np.linspace(-30.0, 30.0, 2001),
    Voff_mV=cache.offsetanalysis.Voff_mV,
    Ioff_nA=cache.offsetanalysis.Ioff_nA,
    cutoff_Hz=13.7,
    sampling_Hz=1370.0,
    median_bins=3,
    sigma_bins=2.0,
)

cache.samplingspec.keys()

cache.exp_v, cache.exp_i = sample(
    traces=cache.traces,
    samplingspec=cache.samplingspec,
)
_ = save_cache(cache)

sampling:   0%|          | 0/143 [00:00<?, ?trace/s]

In [ ]:
# save data
import numpy as np

np.savez_compressed(
    "amp_18.3GHz/eva.npz",
    Vbias_mV=cache.exp_v["V_mV"],
    Ibias_nA=cache.exp_i["I_nA"],
    Aout_mV=cache.exp_v["Aout_mV"],
    Tbath_K=cache.traces.Tsample_K,
    nu_GHz=18.3,
    dGexp_G0=cache.exp_v["dG_G0"],
    dRexp_R0=cache.exp_i["dR_R0"],
    Iexp_nA=cache.exp_v["I_nA"],
)

# Amplitude Study 13.6 GHz

In [20]:
# # generate cache
# from superconductivity.utilities import make_cache, save_cache

# cache = make_cache(name="cache", path="amp_13.6GHz/")
# save_cache(cache)

In [21]:
# load cache
from superconductivity.utilities import load_cache, save_cache

cache = load_cache("cache", path="amp_13.6GHz/")

In [22]:
# get data
from superconductivity.evaluation import FileSpec

cache.filespec = FileSpec(
    h5path="OI-25c-09 2025-05-02 unbroken stripline irradiation studies 0.hdf5",
    location="/Users/oliver/Documents/measurement data/25 04 OI-25c-09",
    measurement="vna_amplitudes_13.6000GHz",
)
mkeys = cache.filespec.mkeys()
skeys = cache.filespec.skeys()
_ = save_cache(cache)

In [23]:
# get keys
from superconductivity.evaluation import KeysSpec
from superconductivity.evaluation import get_keys

cache.keysspec = KeysSpec(
    strip0="GHz_",
    strip1="V",
    remove_key="no_irradiation",
    add_key=[
        ("no_irradiation", 0.0),
        ("no_irradiation", 0.005),
    ],
    limits=(None, None),
    norm=1e-3,
    label="Aout_mV",
)

cache.skeys = get_keys(
    filespec=cache.filespec,
    keysspec=cache.keysspec,
)
_ = save_cache(cache)

In [24]:
# get traces
from superconductivity.evaluation import TraceSpec
from superconductivity.evaluation import get_traces

cache.tracespec = TraceSpec(
    AmpV=1000,
    AmpI=1000,
    trigger_values=1,
    skip_edges=5,
)

cache.traces = get_traces(
    filespec=cache.filespec,
    keysspec=cache.keysspec,
    tracespec=cache.tracespec,
)
_ = save_cache(cache)

In [25]:
# # offset analysis
# from superconductivity.evaluation import OffsetSpec
# from superconductivity.evaluation import offset_analysis
# import numpy as np

# cache.offsetspec = OffsetSpec(
#     Vbins_mV=np.linspace(-0.5, 0.5, 51),
#     Ibins_nA=np.linspace(-5.0, 5.0, 51),
#     Voffscan_mV=np.linspace(-0.045, 0.045, 451),
#     Ioffscan_nA=np.linspace(-0.35, 0.35, 351),
#     cutoff_Hz=13.7,
#     sampling_Hz=137.0,
# )

# cache.offsetanalysis = offset_analysis(
#     traces=cache.traces,
#     spec=cache.offsetspec,
# )
# _ = save_cache(cache)

In [26]:
# sampling
from superconductivity.evaluation import SamplingSpec
from superconductivity.evaluation import sample
import numpy as np

cache.samplingspec = SamplingSpec(
    Vbins_mV=np.linspace(-1.8, 1.8, 1801),
    Ibins_nA=np.linspace(-30.0, 30.0, 2001),
    Voff_mV=cache.offsetanalysis.Voff_mV,
    Ioff_nA=cache.offsetanalysis.Ioff_nA,
    cutoff_Hz=13.7,
    sampling_Hz=137.0,
    median_bins=3,
    sigma_bins=2.0,
)

cache.samplingspec.keys()

cache.exp_v, cache.exp_i = sample(
    traces=cache.traces,
    samplingspec=cache.samplingspec,
)
_ = save_cache(cache)

sampling:   0%|          | 0/201 [00:00<?, ?trace/s]

In [29]:
# save data
import numpy as np

np.savez_compressed(
    "amp_13.6GHz/eva.npz",
    Vbias_mV=cache.exp_v["V_mV"],
    Ibias_nA=cache.exp_i["I_nA"],
    Aout_mV=cache.exp_v["Aout_mV"],
    Tbath_K=cache.traces.Tsample_K,
    nu_GHz=13.6,
    dGexp_G0=cache.exp_v["dG_G0"],
    dRexp_R0=cache.exp_i["dR_R0"],
    Iexp_nA=cache.exp_v["I_nA"],
)

# Frequency Study (0.2V) Study

In [ ]:
# # generate cache
# from superconductivity.utilities import make_cache, save_cache

# cache = make_cache(name="cache", path="nu_0.2V/")
# save_cache(cache)

In [ ]:
# load cache
from superconductivity.utilities import load_cache, save_cache

cache = load_cache("cache", path="nu_0.2V/")

In [ ]:
# get data
from superconductivity.evaluation import FileSpec

cache.filespec = FileSpec(
    h5path="OI-25c-09 2025-05-02 unbroken stripline irradiation studies 0.hdf5",
    location="/Users/oliver/Documents/measurement data/25 04 OI-25c-09",
    measurement="vna_frequencies_0.200V",
)
mkeys = cache.filespec.mkeys()
skeys = cache.filespec.skeys()
_ = save_cache(cache)

In [ ]:
# get keys
from superconductivity.evaluation import KeysSpec
from superconductivity.evaluation import get_keys

cache.keysspec = KeysSpec(
    strip0="vna_",
    strip1="GHz_0.200V",
    remove_key="no_irradiation",
    add_key=[
        ("no_irradiation", 0.0),
    ],
    limits=(None, None),
    norm=1,
    label="nu_GHz",
)

cache.skeys = get_keys(
    filespec=cache.filespec,
    keysspec=cache.keysspec,
)
_ = save_cache(cache)

In [ ]:
# get traces
from superconductivity.evaluation import TraceSpec
from superconductivity.evaluation import get_traces

cache.tracespec = TraceSpec(
    AmpV=1000,
    AmpI=1000,
    trigger_values=1,
    skip_edges=5,
)

cache.traces = get_traces(
    filespec=cache.filespec,
    keysspec=cache.keysspec,
    tracespec=cache.tracespec,
)
_ = save_cache(cache)

In [ ]:
# # offset analysis
# from superconductivity.evaluation import OffsetSpec
# from superconductivity.evaluation import offset_analysis
# import numpy as np

# cache.offsetspec = OffsetSpec(
#     Vbins_mV=np.linspace(-0.5, 0.5, 51),
#     Ibins_nA=np.linspace(-5.0, 5.0, 51),
#     Voffscan_mV=np.linspace(-0.045, 0.045, 451),
#     Ioffscan_nA=np.linspace(-0.35, 0.35, 351),
#     cutoff_Hz=43.7,
#     sampling_Hz=137.0,
# )

# cache.offsetanalysis = offset_analysis(
#     traces=cache.traces,
#     spec=cache.offsetspec,
# )
# _ = save_cache(cache)

In [ ]:
# sampling

from superconductivity.evaluation import SamplingSpec
from superconductivity.evaluation import sample
import numpy as np

cache.samplingspec = SamplingSpec(
    Vbins_mV=np.linspace(-1.6, 1.6, 1601),
    Ibins_nA=np.linspace(-30.0, 30.0, 2001),
    Voff_mV=cache.offsetanalysis.Voff_mV,
    Ioff_nA=cache.offsetanalysis.Ioff_nA,
    cutoff_Hz=13.7,
    sampling_Hz=137.0,
    median_bins=3,
    sigma_bins=2.0,
)

cache.samplingspec.keys()

cache.exp_v, cache.exp_i = sample(
    traces=cache.traces,
    samplingspec=cache.samplingspec,
)
_ = save_cache(cache)

In [ ]:
# save data
import numpy as np

np.savez_compressed(
    "nu_0.2V/eva.npz",
    Vbias_mV=cache.exp_v["V_mV"],
    Ibias_nA=cache.exp_i["I_nA"],
    nu_GHz=cache.exp_v["nu_GHz"],
    Aout_mV=0.2,
    dGexp_G0=cache.exp_v["dG_G0"],
    dRexp_R0=cache.exp_i["dR_R0"],
    Iexp_nA=cache.exp_v["I_nA"],
)

# Frequency Study (0.1V) Study

In [ ]:
# # generate cache
# from superconductivity.utilities import make_cache, save_cache

# cache = make_cache(name="cache", path="nu_0.1V/")
# save_cache(cache)

In [ ]:
# load cache
from superconductivity.utilities import load_cache, save_cache

cache = load_cache("cache", path="nu_0.1V/")

In [ ]:
# get data
from superconductivity.evaluation import FileSpec

cache.filespec = FileSpec(
    h5path="OI-25c-09 2025-05-02 unbroken stripline irradiation studies 0.hdf5",
    location="/Users/oliver/Documents/measurement data/25 04 OI-25c-09",
    measurement="vna_frequencies_0.100V",
)
mkeys = cache.filespec.mkeys()
skeys = cache.filespec.skeys()
_ = save_cache(cache)

In [ ]:
# get keys
from superconductivity.evaluation import KeysSpec
from superconductivity.evaluation import get_keys

cache.keysspec = KeysSpec(
    strip0="vna_",
    strip1="GHz_0.100V",
    remove_key="no_irradiation",
    add_key=[
        ("no_irradiation", 0.0),
    ],
    limits=(None, None),
    norm=1,
    label="nu_GHz",
)

cache.skeys = get_keys(
    filespec=cache.filespec,
    keysspec=cache.keysspec,
)
_ = save_cache(cache)

In [ ]:
# get traces
from superconductivity.evaluation import TraceSpec
from superconductivity.evaluation import get_traces

cache.tracespec = TraceSpec(
    AmpV=1000,
    AmpI=1000,
    trigger_values=1,
    skip_edges=5,
)

cache.traces = get_traces(
    filespec=cache.filespec,
    keysspec=cache.keysspec,
    tracespec=cache.tracespec,
)
_ = save_cache(cache)

In [ ]:
# # offset analysis
# from superconductivity.evaluation import OffsetSpec
# from superconductivity.evaluation import offset_analysis
# import numpy as np

# cache.offsetspec = OffsetSpec(
#     Vbins_mV=np.linspace(-0.5, 0.5, 51),
#     Ibins_nA=np.linspace(-5.0, 5.0, 51),
#     Voffscan_mV=np.linspace(-0.045, 0.045, 451),
#     Ioffscan_nA=np.linspace(-0.35, 0.35, 351),
#     cutoff_Hz=43.7,
#     sampling_Hz=137.0,
# )

# cache.offsetanalysis = offset_analysis(
#     traces=cache.traces,
#     spec=cache.offsetspec,
# )
# _ = save_cache(cache)

In [ ]:
# sampling

from superconductivity.evaluation import SamplingSpec
from superconductivity.evaluation import sample
import numpy as np

cache.samplingspec = SamplingSpec(
    Vbins_mV=np.linspace(-1.6, 1.6, 1601),
    Ibins_nA=np.linspace(-30.0, 30.0, 2001),
    Voff_mV=cache.offsetanalysis.Voff_mV,
    Ioff_nA=cache.offsetanalysis.Ioff_nA,
    cutoff_Hz=13.7,
    sampling_Hz=137.0,
    median_bins=3,
    sigma_bins=2.0,
)

cache.samplingspec.keys()

cache.exp_v, cache.exp_i = sample(
    traces=cache.traces,
    samplingspec=cache.samplingspec,
)
_ = save_cache(cache)

In [ ]:
# save data
import numpy as np

np.savez_compressed(
    "nu_0.1V/eva.npz",
    Vbias_mV=cache.exp_v["V_mV"],
    Ibias_nA=cache.exp_i["I_nA"],
    nu_GHz=cache.exp_v["nu_GHz"],
    Aout_mV=0.1,
    dGexp_G0=cache.exp_v["dG_G0"],
    dRexp_R0=cache.exp_i["dR_R0"],
    Iexp_nA=cache.exp_v["I_nA"],
)

# Quick Checks

In [ ]:
# load all cache
import matplotlib.pyplot as plt
import numpy as np
import superconductivity as sc
from superconductivity.utilities import load_cache

temperature = load_cache("cache", path="temperature/")
amp_183GHz = load_cache("cache", path="amp_18.3GHz/")
amp_136GHz = load_cache("cache", path="amp_13.6GHz/")
nu_01V = load_cache("cache", path="nu_0.1V/")
nu_02V = load_cache("cache", path="nu_0.2V/")